# Frankfurt Bike-Sharing: filtered full dataset viewer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/data_loaders/frankfurt_bike_sharing/notebooks/frankfurt_bike_sharing_gcn_loader.ipynb)

This notebook reads the locally filtered Frankfurt subset from the TUM FTM European Bike-Sharing Dataset full export. It is read-only: it does not download the original 900 MB+ full archive and does not create new bike-sharing snapshots.

Use the filtered zip file `frankfurt_bike_sharing_full_filtered.zip`. In Colab, either paste a direct Seafile download link below or upload the zip when prompted.

In [ ]:
!pip -q install pandas folium plotly

In [ ]:
from pathlib import Path
import zipfile
import urllib.request

import folium
import pandas as pd
import plotly.express as px
from IPython.display import display

pd.set_option("display.max_columns", 100)

## 1. Load the filtered Frankfurt zip

The large source `full/dataset.zip` stays on your local computer and is not committed to GitHub. This notebook expects the much smaller filtered Frankfurt package.

In [ ]:
ZIP_NAME = "frankfurt_bike_sharing_full_filtered.zip"

# Seafile file link for the filtered Frankfurt zip.
# If this link expires, upload the same zip manually when Colab prompts you.
SEAFILE_ZIP_URL = "https://seafile.rlp.net/seafhttp/f/81218a2c4f2c4ee28824/?op=view"

ROOT = Path.cwd()
if not (ROOT / "data_loaders" / "frankfurt_bike_sharing").exists():
    candidate = Path("/content/KI_Geodatenanalyse_SS26")
    if (candidate / "data_loaders" / "frankfurt_bike_sharing").exists():
        ROOT = candidate

candidate_paths = [
    ROOT / "data_loaders" / "frankfurt_bike_sharing" / "data" / ZIP_NAME,
    Path("/content") / ZIP_NAME,
    Path.cwd() / ZIP_NAME,
]


def seafile_download_candidates(url):
    candidates = [url]
    if "op=view" in url:
        candidates.append(url.replace("op=view", "op=download"))
    if "dl=1" not in url:
        sep = "&" if "?" in url else "?"
        candidates.append(f"{url}{sep}dl=1")
    return list(dict.fromkeys(candidates))


def download_zip_from_seafile(url, target):
    last_error = None
    for download_url in seafile_download_candidates(url):
        try:
            print("Trying:", download_url)
            urllib.request.urlretrieve(download_url, target)
            if zipfile.is_zipfile(target):
                return target
            last_error = RuntimeError("Downloaded file is not a valid zip.")
        except Exception as exc:
            last_error = exc
    if target.exists():
        target.unlink()
    raise RuntimeError(f"Could not download a valid zip from Seafile: {last_error}")


zip_path = next((path for path in candidate_paths if path.exists() and zipfile.is_zipfile(path)), None)
if zip_path is None and SEAFILE_ZIP_URL:
    zip_path = Path("/content") / ZIP_NAME if Path("/content").exists() else Path.cwd() / ZIP_NAME
    zip_path = download_zip_from_seafile(SEAFILE_ZIP_URL, zip_path)

if zip_path is None:
    try:
        from google.colab import files

        print(f"Upload {ZIP_NAME} now.")
        uploaded = files.upload()
        if ZIP_NAME not in uploaded:
            raise FileNotFoundError(f"Uploaded files do not include {ZIP_NAME}.")
        zip_path = Path("/content") / ZIP_NAME
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            f"Could not find {ZIP_NAME}. Put it in one of these locations: {candidate_paths}"
        ) from exc

extract_dir = Path("/content/frankfurt_bike_sharing_full") if Path("/content").exists() else zip_path.parent / "frankfurt_bike_sharing_full"
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_dir)

data_dir = extract_dir / "frankfurt_full"
print("Zip:", zip_path)
print("Extracted data:", data_dir)
print("Files:")
for path in sorted(data_dir.glob("*")):
    print(" -", path.name)

## 2. Read station metadata and time series

In [ ]:
stations = pd.read_csv(data_dir / "stations_frankfurt.csv")
station_status = pd.read_csv(data_dir / "station_status_frankfurt.csv")
trips = pd.read_csv(data_dir / "trips_frankfurt.csv")

stations["id"] = stations["id"].astype(str)
station_status["station_id"] = station_status["station_id"].astype(str)
trips["station_id_start"] = trips["station_id_start"].astype("Int64").astype(str).replace("<NA>", pd.NA)
trips["station_id_end"] = trips["station_id_end"].astype("Int64").astype(str).replace("<NA>", pd.NA)

station_status["datetime_utc"] = pd.to_datetime(station_status["time"], unit="s", utc=True)
station_status["datetime_berlin"] = station_status["datetime_utc"].dt.tz_convert("Europe/Berlin")
trips["datetime_start_utc"] = pd.to_datetime(trips["time_start"], unit="s", utc=True)
trips["datetime_start_berlin"] = trips["datetime_start_utc"].dt.tz_convert("Europe/Berlin")

print(f"Stations: {len(stations):,}")
print(f"Station-status rows: {len(station_status):,}")
print(f"Trips: {len(trips):,}")
print(
    "Station-status time range:",
    station_status["datetime_berlin"].min(),
    "to",
    station_status["datetime_berlin"].max(),
)

display(stations.head())
display(station_status.head())

## 3. Raw time series for the top 3 stations

The top 3 stations are selected by the number of station-status observations in the selected month. The plot uses the original observations without resampling. A step line (`hv`) shows the last observed value as unchanged until the next recorded update.

In [ ]:
# Change this month if you want to inspect another part of the dataset.
MONTH_START = "2022-09-01"
month_start = pd.Timestamp(MONTH_START, tz="Europe/Berlin")
month_end = month_start + pd.DateOffset(months=1)

status_month = station_status[
    (station_status["datetime_berlin"] >= month_start)
    & (station_status["datetime_berlin"] < month_end)
].copy()

if status_month.empty:
    raise ValueError(f"No station-status observations found for {month_start:%Y-%m}.")

top3_stations = (
    status_month.groupby("station_id", as_index=False)
    .agg(status_observations=("time", "count"))
    .sort_values("status_observations", ascending=False)
    .head(3)
    .merge(stations[["id", "name"]], left_on="station_id", right_on="id", how="left")
    [["station_id", "name", "status_observations"]]
)
top3_ids = top3_stations["station_id"].tolist()

ts = status_month[status_month["station_id"].isin(top3_ids)].merge(
    top3_stations[["station_id", "name"]],
    on="station_id",
    how="left",
)
ts = ts.sort_values(["name", "datetime_berlin"])

print(f"Selected month: {month_start:%Y-%m}")
print("Top 3 stations by station-status observations in this month:")
display(top3_stations)

fig = px.line(
    ts,
    x="datetime_berlin",
    y="bikes_available_to_rent",
    color="name",
    line_shape="hv",
    markers=True,
    title="Raw available-bike observations for the top 3 Frankfurt stations",
    labels={
        "datetime_berlin": "Time (Europe/Berlin)",
        "bikes_available_to_rent": "Available bikes",
        "name": "Station",
    },
    height=560,
)
fig.update_traces(marker={"size": 4})
fig.show()

display(
    ts.groupby(["station_id", "name"], as_index=False)
    .agg(
        observations=("time", "count"),
        first_observation=("datetime_berlin", "min"),
        last_observation=("datetime_berlin", "max"),
        mean_available_bikes=("bikes_available_to_rent", "mean"),
        max_available_bikes=("bikes_available_to_rent", "max"),
    )
)

## 4. Station spatial distribution

In [ ]:
station_counts = station_status.groupby("station_id", as_index=False).agg(
    status_observations=("time", "count"),
    mean_available_bikes=("bikes_available_to_rent", "mean"),
)
station_map_data = stations.merge(station_counts, left_on="id", right_on="station_id", how="left")
station_map_data["status_observations"] = station_map_data["status_observations"].fillna(0).astype(int)
station_map_data["mean_available_bikes"] = station_map_data["mean_available_bikes"].fillna(0)

m = folium.Map(
    location=[station_map_data["lat"].mean(), station_map_data["lon"].mean()],
    zoom_start=12,
    tiles="cartodbpositron",
)

for _, row in station_map_data.iterrows():
    popup = folium.Popup(
        f"<b>{row['name']}</b><br>"
        f"Station id: {row['id']}<br>"
        f"Racks: {row['bike_racks']}<br>"
        f"Status observations: {row['status_observations']}<br>"
        f"Mean available bikes: {row['mean_available_bikes']:.2f}",
        max_width=320,
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4 + min(row["status_observations"], 3000) / 1000,
        color="#1f78b4",
        fill=True,
        fill_color="#1f78b4",
        fill_opacity=0.75,
        weight=1,
        tooltip=f"{row['name']} ({row['status_observations']} observations)",
        popup=popup,
    ).add_to(m)

m

## 5. Optional trip activity time series

In [ ]:
trips_month = trips[
    (trips["datetime_start_berlin"] >= month_start)
    & (trips["datetime_start_berlin"] < month_end)
].copy()

trips_daily = (
    trips_month.set_index("datetime_start_berlin")
    .resample("1D")
    .size()
    .rename("trips")
    .reset_index()
)

fig_trips = px.line(
    trips_daily,
    x="datetime_start_berlin",
    y="trips",
    markers=True,
    title="Daily Frankfurt bike-sharing trips in the selected month",
    labels={"datetime_start_berlin": "Date (Europe/Berlin)", "trips": "Trips"},
    height=420,
)
fig_trips.show()